In [0]:
select distinct code from tx_claim

In [0]:
CREATE OR REPLACE TEMPORARY VIEW specialty_groupings AS
SELECT * FROM (
    VALUES
        ('Psyciatry and Neurology', 'Psychiatry & Neurology'), 
        ('Psyciatry and Neurology', 'Neurological Surgery'), 
        ('Others', 'Surgery'), 
        ('Geneticist', 'Medical Genetics'), 
        ('Pediatrician', 'Pediatrics'), 
        ('PCP', 'Family Medicine'), 
        ('PCP', 'Internal Medicine'), 
        ('NP/PA', 'Physician Assistant'), 
        ('NP/PA', 'Nurse Practitioner'), 
        ('Others', 'Orthopaedic Surgery'), 
        ('Others', 'Otolaryngology'), 
        ('Others', 'Ophthalmology'), 
        ('NP/PA', 'Nurse Anesthetist, Certified Registered'), 
        ('Others', 'Anesthesiology'), 
        ('Others', 'Radiology'), 
        ('Others', 'Pathology'), 
        ('Others', 'Emergency Medicine'), 
        ('Others', 'Hospitalist'), 
        ('Others', 'Physical Medicine & Rehabilitation')
) AS t(mapped_bucket, specialty);


In [0]:
-- Elaprase Treated patients (Can or cannot be mpsii diagnosed)
-- A single patient can be attributed to multiple HCPs

create or replace temporary view tx_claims as
with elaprase_tx_claims as (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 RENDERING_NPI AS NPI,
                 REFERRING_NPI AS REFERRING_NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NULL AS REFERRING_NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                REFERRING_NPI AS REFERRING_NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')),
specialty_info as (
  select a.*, b.PRIMARY_SPECIALTY as specialty, 
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'other'
            ELSE c.mapped_bucket
  END AS specialty_bucket,
  pos.description as pos_description, d.HCO_PRIMARY_NPI as hco_npi_thm
  from elaprase_tx_claims as a
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
),
tx_claims_5_years as (
  select *
from specialty_info
where fill_date between '2020-08-01' and '2025-07-31'
),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM tx_claims_5_years a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi)
SELECT
  -- explicit columns from tx_claims_2_years (add/remove columns here as needed)
  a.PATIENT_ID,
  a.NPI,
  a.referring_npi,
  a.CODE,
  a.EVENT_ID,
  a.FILL_DATE,
  a.PLACE_OF_SERVICE,
  a.KH_PLAN,
  a.TABLE_NAME,
  a.specialty,
  a.specialty_bucket,
  a.pos_description,
  COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM tx_claims_5_years a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi;

In [0]:
create or replace temporary view dx_claims as
with diagnosis_claims as (
  SELECT DISTINCT 
      PATIENT_ID,
      RENDERING_NPI AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES ilike '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE ILIKE '%E761%'
      AND TRANSACTION_STATUS = 'PAID'),
specialty_info as (
  select a.*, b.primary_specialty as specialty, 
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'other'
            ELSE c.mapped_bucket
  END AS specialty_bucket, 
  pos.description as pos_description, d.HCO_PRIMARY_NPI as hco_npi_thm
  from diagnosis_claims as a
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.npi and b.provider_type = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
),
-- dx_claims_5_years as (select * from specialty_info
-- where fill_date between '2020-08-01' AND '2025-07-31'),
-- Now filtering data for 5 years and keeping only those patients who are having >=2 Dx claims of E761
dx_claims_5_years as (select * from specialty_info
where patient_id in (
  select patient_id from specialty_info
  where fill_date between '2020-08-01' AND '2025-07-31'
  group by patient_id having count(distinct fill_date) >= 2
) and fill_date between '2020-08-01' AND '2025-07-31'),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM dx_claims_5_years a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi)
SELECT
  -- explicit columns from tx_claims_2_years (add/remove columns here as needed)
  a.PATIENT_ID,
  a.NPI,
  a.FILL_DATE,
  a.EVENT_ID,
  a.DIAGNOSIS_CODES,
  a.KH_PLAN,
  a.PLACE_OF_SERVICE,
  a.specialty,
  a.specialty_bucket,
  a.pos_description,
  COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM dx_claims_5_years a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi;

In [0]:
create or replace temporary view tx_claims_other_elaprase as
with all_tx_claims as (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 RENDERING_NPI AS NPI,
                 REFERRING_NPI AS REFERRING_NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
        --  WHERE NDC11 IN ('54092070001','540920700')
        -- where DIAGNOSIS_CODES ilike '%E761%'
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NULL AS REFERRING_NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE 
           
          -- NDC11 IN ('54092070001','540920700')
          --  AND 
           TRANSACTION_RESULT = 'PAID'
          --  and DIAGNOSIS_CODE ilike '%E761%'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                REFERRING_NPI AS REFERRING_NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
          --  WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          -- where DIAGNOSIS_CODES ilike '%E761%'
           ),
specialty_info as (
  select a.*, b.PRIMARY_SPECIALTY as specialty, 
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'other'
            ELSE c.mapped_bucket
  END AS specialty_bucket, 
  pos.description as pos_description, d.HCO_PRIMARY_NPI as hco_npi_thm
  from all_tx_claims as a 
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
   where a.patient_id in (select distinct patient_id from dx_claims) and a.patient_id not in (select distinct patient_id from tx_claims)
),
tx_claims_5_years as (
  select *
from specialty_info
where fill_date between '2020-08-01' and '2025-07-31'
),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM tx_claims_5_years a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi)
SELECT
  -- explicit columns from tx_claims_2_years (add/remove columns here as needed)
  a.PATIENT_ID,
  a.NPI,
  a.referring_npi,
  a.CODE,
  a.EVENT_ID,
  a.FILL_DATE,
  a.PLACE_OF_SERVICE,
  a.KH_PLAN,
  a.TABLE_NAME,
  a.specialty,
  a.specialty_bucket,
  a.pos_description,
  COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM tx_claims_5_years a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi;

In [0]:
-- Elaprase Treated patients (Can or cannot be mpsii diagnosed)
-- A single patient can be attributed to multiple HCPs

create or replace temporary view mpsii_diagnosed_elaprase_treated_claims as
with mpsii_diagnosed_elaprase_tx_claims as (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 RENDERING_NPI AS NPI,
                 REFERRING_NPI AS REFERRING_NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NULL AS REFERRING_NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                REFERRING_NPI AS REFERRING_NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')),
specialty_info as (
  select a.*, b.PRIMARY_SPECIALTY as specialty, 
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'Others'
            ELSE c.mapped_bucket
  END AS specialty_bucket, 
  pos.description as pos_description, d.HCO_PRIMARY_NPI as hco_npi_thm
  from mpsii_diagnosed_elaprase_tx_claims as a 
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
  where a.patient_id in (select distinct patient_id from dx_claims)
),
tx_claims_5_years as (
  select *
from specialty_info
where fill_date between '2020-08-01' and '2025-07-31'
),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM tx_claims_5_years a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi)
SELECT
  -- explicit columns from tx_claims_2_years (add/remove columns here as needed)
  a.PATIENT_ID,
  a.NPI,
  a.referring_npi,
  a.CODE,
  a.EVENT_ID,
  a.FILL_DATE,
  a.PLACE_OF_SERVICE,
  a.KH_PLAN,
  a.TABLE_NAME,
  a.specialty,
  a.specialty_bucket,
  a.pos_description,
  COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM tx_claims_5_years a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi;

### Primary NPI Tagging

In [0]:
-- select count(distinct patient_id) from tx_claims
-- select * from tx_claims limit 4;

In [0]:
Create or replace temp view tx_claims_v2 AS
with primary_npi as (
  select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    HCO_PRIMARY_NPI,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        HCO_PRIMARY_NPI,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        HCO_PRIMARY_NPI,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            HCO_PRIMARY_NPI,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select patient_id, npi, fill_date from tx_claims) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,HCO_PRIMARY_NPI
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
),
  primary_npi_flag as (SELECT 
    a.*,
    CASE 
        WHEN b.primary_npi IS NOT NULL THEN 1 
        ELSE 0 
    END AS is_primary
FROM tx_claims AS a
LEFT JOIN primary_npi AS b
    ON a.patient_id = b.patient_id 
   AND a.npi = b.primary_npi)
select * from primary_npi_flag where is_primary = 1;

In [0]:
Create or replace temp view dx_claims_v2 AS
with primary_npi as (
  select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    HCO_PRIMARY_NPI,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        HCO_PRIMARY_NPI,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        HCO_PRIMARY_NPI,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            HCO_PRIMARY_NPI,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select patient_id, npi, fill_date from dx_claims) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,HCO_PRIMARY_NPI
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
),
  primary_npi_flag as (SELECT 
    a.*,
    CASE 
        WHEN b.primary_npi IS NOT NULL THEN 1 
        ELSE 0 
    END AS is_primary
FROM dx_claims AS a
LEFT JOIN primary_npi AS b
    ON a.patient_id = b.patient_id 
   AND a.npi = b.primary_npi)
select * from primary_npi_flag where is_primary = 1;

In [0]:
Create or replace temp view tx_claims_other_elaprase_v2 AS
with primary_npi as (
  select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    HCO_PRIMARY_NPI,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        HCO_PRIMARY_NPI,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        HCO_PRIMARY_NPI,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            HCO_PRIMARY_NPI,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select patient_id, npi, fill_date from tx_claims_other_elaprase) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,HCO_PRIMARY_NPI
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
),
  primary_npi_flag as (SELECT 
    a.*,
    CASE 
        WHEN b.primary_npi IS NOT NULL THEN 1 
        ELSE 0 
    END AS is_primary
FROM tx_claims_other_elaprase AS a
LEFT JOIN primary_npi AS b
    ON a.patient_id = b.patient_id 
   AND a.npi = b.primary_npi)
select * from primary_npi_flag where is_primary = 1;

In [0]:
Create or replace temp view mpsii_diagnosed_elaprase_treated_claims_v2 AS
with primary_npi as (
  select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    HCO_PRIMARY_NPI,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        HCO_PRIMARY_NPI,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        HCO_PRIMARY_NPI,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            HCO_PRIMARY_NPI,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2
                                WHEN SPECIALTY = 'Pediatrician' THEN 3
                                WHEN SPECIALTY = 'PCP' THEN 4
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select patient_id, npi, fill_date from mpsii_diagnosed_elaprase_treated_claims) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,HCO_PRIMARY_NPI
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
),
  primary_npi_flag as (SELECT 
    a.*,
    CASE 
        WHEN b.primary_npi IS NOT NULL THEN 1 
        ELSE 0 
    END AS is_primary
FROM mpsii_diagnosed_elaprase_treated_claims AS a
LEFT JOIN primary_npi AS b
    ON a.patient_id = b.patient_id 
   AND a.npi = b.primary_npi)
select * from primary_npi_flag where is_primary = 1;

In [0]:
create or replace temporary view hcp_universe as
select distinct npi
from (select distinct npi from tx_claims_v2
union
select distinct npi from dx_claims_v2
union
select distinct npi from tx_claims_other_elaprase_v2
union
select distinct npi from mpsii_diagnosed_elaprase_treated_claims_v2)

In [0]:
select count(distinct npi) from hcp_universe

# Pulling the Numbers

In [0]:
with elaprase_treatment as (
  select npi, count(distinct patient_id) as elaprase_treated_patients
  from tx_claims_v2 group by 1 order by 2 desc
),
mpsii_diagnosed as (
  select npi, count(distinct patient_id) as mpsii_diagnosed_patients
  from dx_claims_v2 group by 1 order by 2 desc
),
mpsii_treated as (
  select npi, count(distinct patient_id) as mpsii_treated_patients
  from tx_claims_other_elaprase_v2 group by 1 order by 2 desc
),
mpsii_elaprase_treated as (
  select npi, count(distinct patient_id) as mpsii_elaprase_treated_patients
  from mpsii_diagnosed_elaprase_treated_claims_v2 group by 1 order by 2 desc
),
joining_patient_numbers as (
  select a.*, b.elaprase_treated_patients, c.mpsii_diagnosed_patients, d.mpsii_treated_patients, e.mpsii_elaprase_treated_patients
  from hcp_universe as a
  left join elaprase_treatment as b on a.npi = b.npi
  left join mpsii_diagnosed as c on a.npi = c.npi
  left join mpsii_treated as d on a.npi = d.npi
  left join mpsii_elaprase_treated as e on a.npi = e.npi
),
kol_list AS (
  SELECT * FROM VALUES
    ('1699743088'),('1154431567'),('1467848366'),('1528585833'),('1114949617'),('1861866717'),('1942545314'),('1255435301'),('1104395656'),('1770949901'),('1134534597'),('1831455690'),('1003203779'),('1477999522'),('1902012180'),('1114360997'),('1104906445'),('1740218296'),('1073711966'),('1588735005'),('1780931956'),('1124319462'),('1215983382')
  AS kol(npi)
),
-- Step 2: Combine your HCP info with KOL list
tagged_hcps AS (
  SELECT
      a.*,
      CASE WHEN b.npi IS NOT NULL THEN 1 ELSE 0 END AS is_kol
  FROM joining_patient_numbers a
  LEFT JOIN kol_list b
  ON a.npi = b.npi

  UNION ALL

  -- Step 3: Add KOLs missing from your main table
  SELECT
      b.npi,
      NULL AS elaprase_treated_patients,
      NULL AS mpsii_diagnosed_patients,
      NULL AS mpsii_treated_patients,
      NULL AS mpsii_elaprase_treated_patients,
      1 AS is_kol
  FROM kol_list b
  LEFT JOIN joining_patient_numbers a
  ON a.npi = b.npi
  WHERE a.npi IS NULL
),
hcp_info as (
  select a.*, concat(b.FIRST_NAME, " ", b.LAST_NAME) as hcp_name, b.PRIMARY_SPECIALTY as specialty,
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'Others'
            ELSE c.mapped_bucket
  END AS specialty_bucket, d.HCO_PRIMARY_NPI as hco_npi_thm
  from tagged_hcps as a
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM hcp_info a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi),
hcp_info_with_hco_info as (
  SELECT
  a.npi, a.elaprase_treated_patients, a.mpsii_diagnosed_patients, a.mpsii_treated_patients, a.mpsii_elaprase_treated_patients, a.hcp_name, a.specialty, a.specialty_bucket, COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM hcp_info a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi
),
hco_type_info_added as (
  select a.*, c.name as hco_type_info
  from hcp_info_with_hco_info as a
  left join com_edp_prd.com_raw.vod_hco as b
on a.primary_hco = try_cast(b.npi_num__v as BIGINT)
left join com_edp_prd.com_raw.vod_references as c
on b.hco_type__v = c.code and c.reference_type = 'HCOType'
)
select * from hco_type_info_added;

### Updating Distributions

In [0]:
select * 
from mpsii_diagnosed_elaprase_treated_claims_v2

In [0]:
with hco_type_info as (select a.*, b.hco_type__v as hco_type, c.name as hco_type_info
from mpsii_diagnosed_elaprase_treated_claims_v2 as a
left join com_edp_prd.com_raw.vod_hco as b
on a.primary_hco = try_cast(b.npi_num__v as BIGINT)
left join com_edp_prd.com_raw.vod_references as c
on b.hco_type__v = c.code and c.reference_type = 'HCOType')
select hco_type_info, count(distinct patient_id) as patient_count
from hco_type_info
-- where hco_type_info is not null
group by 1 order by 2 desc

In [0]:
select *
from mpsii_diagnosed_elaprase_treated_claims_v2
where patient_id in (select patient_id
from mpsii_diagnosed_elaprase_treated_claims_v2
group by 1 
having count(distinct place_of_service) >= 2)

In [0]:
select place_of_service ,pos_description, count(distinct patient_id)
from mpsii_diagnosed_elaprase_treated_claims_v2
group by 1,2 order by 3 desc